In [ ]:
# ==========================================
# STEP 1A: Install PyTorch Geometric in Colab
# ==========================================

import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"])

0

In [ ]:
# =========================
# STEP 1: Import libraries
# =========================

import os
import math
import time
import random
import itertools
import warnings
from collections import defaultdict
from functools import lru_cache

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GINConv, GCNConv, GATConv, global_add_pool

In [ ]:
print("PyTorch version:", torch.__version__)
print("PyG version:", __import__("torch_geometric").__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

PyTorch version: 2.11.0+cu128
PyG version: 2.8.0
CUDA available: True
CUDA device: Tesla T4


In [ ]:
# ==========================================
# STEP 2: Reproducibility, device, constants
# ==========================================

SEED = 42

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Reproducibility flags
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# -------------------------
# Graph-generation settings
# -------------------------
N_MIN = 6
N_MAX = 30

NUM_UNICYCLIC_PER_ORDER = 200
NUM_BICYCLIC_PER_ORDER = 200
MAX_TRANSFORMS_PER_GRAPH = 2

# -------------------------
# Data split settings
# -------------------------
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# -------------------------
# Node feature settings
# [degree, cycle_membership, pendant_neighbors, leaf_indicator, eccentricity]
# -------------------------
NUM_NODE_FEATURES = 5

# -------------------------
# Training settings
# -------------------------
HIDDEN_DIM = 64
DROPOUT = 0.20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
BATCH_SIZE = 64
MAX_EPOCHS = 200
EARLY_STOPPING_PATIENCE = 20

# -------------------------
# Output settings
# [W, MS, Z, M1, M2, R]
# -------------------------
TARGET_NAMES = ["W", "MS", "Z", "M1", "M2", "R"]
NUM_TARGETS = len(TARGET_NAMES)

print("Target names:", TARGET_NAMES)
print("Graph orders:", N_MIN, "to", N_MAX)
print("Transforms per graph:", MAX_TRANSFORMS_PER_GRAPH)

Using device: cuda
Target names: ['W', 'MS', 'Z', 'M1', 'M2', 'R']
Graph orders: 6 to 30
Transforms per graph: 2


In [ ]:
# ==========================================
# STEP 3: Graph-family validation functions
# ==========================================

def cyclomatic_number(G: nx.Graph) -> int:
    """
    Cyclomatic number for a connected graph:
        mu(G) = m - n + 1
    """
    n = G.number_of_nodes()
    m = G.number_of_edges()
    return m - n + 1


def is_simple_connected_graph(G: nx.Graph) -> bool:
    """
    Returns True if G is a simple connected undirected graph.
    NetworkX Graph objects are simple by construction if we use nx.Graph,
    so we mainly verify connectivity and no self-loops.
    """
    if G.number_of_nodes() == 0:
        return False
    if not nx.is_connected(G):
        return False
    if any(u == v for u, v in G.edges()):
        return False
    return True


def is_unicyclic_graph(G: nx.Graph) -> bool:
    """
    True iff G is simple, connected, and has cyclomatic number 1.
    """
    return is_simple_connected_graph(G) and cyclomatic_number(G) == 1


def is_bicyclic_graph(G: nx.Graph) -> bool:
    """
    True iff G is simple, connected, and has cyclomatic number 2.
    """
    return is_simple_connected_graph(G) and cyclomatic_number(G) == 2


def graph_family_label(G: nx.Graph) -> str:
    """
    Returns 'unicyclic', 'bicyclic', or raises an error.
    """
    if is_unicyclic_graph(G):
        return "unicyclic"
    if is_bicyclic_graph(G):
        return "bicyclic"
    raise ValueError("Graph does not belong to the supported families.")

In [ ]:
# Quick sanity checks

G_cycle = nx.cycle_graph(6)  # unicyclic
print("Cycle graph mu:", cyclomatic_number(G_cycle), "| unicyclic:", is_unicyclic_graph(G_cycle))

G_bi = nx.Graph()
G_bi.add_edges_from([(0,1),(1,2),(2,0),(2,3),(3,4),(4,2)])  # two triangles sharing one vertex
print("Bicyclic graph mu:", cyclomatic_number(G_bi), "| bicyclic:", is_bicyclic_graph(G_bi))

Cycle graph mu: 1 | unicyclic: True
Bicyclic graph mu: 2 | bicyclic: True


In [ ]:
# ==========================================
# STEP 4: Unicyclic graph generator
# ==========================================

def generate_unicyclic_graph(n: int, rng: np.random.Generator = None) -> nx.Graph:
    """
    Generate one random unicyclic graph of order n.

    Procedure:
    1. Sample cycle length l from {3, ..., n}
    2. Construct C_l
    3. Add remaining n-l vertices one at a time
    4. Each new vertex attaches to one existing vertex

    This preserves simplicity, connectivity, and cyclomatic number 1.
    """
    if rng is None:
        rng = np.random.default_rng()

    if n < 3:
        raise ValueError("Unicyclic graph requires n >= 3.")

    # Step 1: sample cycle length
    l = int(rng.integers(3, n + 1))

    # Step 2: construct cycle C_l
    G = nx.cycle_graph(l)

    # Step 3-4: attach remaining vertices one at a time
    next_node = l
    while G.number_of_nodes() < n:
        current_nodes = list(G.nodes())
        attach_to = int(rng.choice(current_nodes))

        G.add_node(next_node)
        G.add_edge(next_node, attach_to)
        next_node += 1

    # Final validation
    if not is_unicyclic_graph(G):
        raise RuntimeError("Generated graph is not unicyclic.")

    return G

In [ ]:
# Quick sanity test for unicyclic generation
rng = np.random.default_rng(42)

for n in [6, 10, 15]:
    G = generate_unicyclic_graph(n, rng)
    print(
        f"n={n}, nodes={G.number_of_nodes()}, edges={G.number_of_edges()}, "
        f"mu={cyclomatic_number(G)}, unicyclic={is_unicyclic_graph(G)}"
    )

In [ ]:
G = generate_unicyclic_graph(10, np.random.default_rng(123))
plt.figure(figsize=(4, 4))
nx.draw(G, with_labels=True, node_size=500)
plt.show()

In [ ]:
# ==========================================
# STEP 5: Bicyclic graph generator
# ==========================================

def make_vertex_sharing_core(l1: int, l2: int) -> nx.Graph:
    """
    Build two cycles sharing exactly one vertex.
    Total nodes = l1 + l2 - 1
    """
    if l1 < 3 or l2 < 3:
        raise ValueError("Cycle lengths must be >= 3.")

    G = nx.Graph()

    # First cycle: 0,1,...,l1-1
    cycle1 = list(range(l1))
    for i in range(l1):
        G.add_edge(cycle1[i], cycle1[(i + 1) % l1])

    # Shared vertex = 0
    # Second cycle: 0 plus new nodes
    start = l1
    cycle2_new = list(range(start, start + l2 - 1))
    cycle2 = [0] + cycle2_new
    for i in range(l2):
        G.add_edge(cycle2[i], cycle2[(i + 1) % l2])

    return G


def make_edge_sharing_core(l1: int, l2: int) -> nx.Graph:
    """
    Build two cycles sharing exactly one edge.
    Total nodes = l1 + l2 - 2
    """
    if l1 < 3 or l2 < 3:
        raise ValueError("Cycle lengths must be >= 3.")

    G = nx.Graph()

    # Shared edge = (0, 1)
    # First cycle uses nodes: 0,1,2,...,l1-1
    cycle1 = list(range(l1))
    for i in range(l1):
        G.add_edge(cycle1[i], cycle1[(i + 1) % l1])

    # Second cycle uses shared edge (0,1) plus new nodes
    start = l1
    cycle2_new = list(range(start, start + l2 - 2))
    cycle2 = [0, 1] + cycle2_new
    for i in range(l2):
        G.add_edge(cycle2[i], cycle2[(i + 1) % l2])

    return G


def make_bridge_connected_core(l1: int, l2: int) -> nx.Graph:
    """
    Build two disjoint cycles connected by a single bridge edge.
    Total nodes = l1 + l2
    Cyclomatic number remains 2.
    """
    if l1 < 3 or l2 < 3:
        raise ValueError("Cycle lengths must be >= 3.")

    G = nx.Graph()

    # First cycle
    cycle1 = list(range(l1))
    for i in range(l1):
        G.add_edge(cycle1[i], cycle1[(i + 1) % l1])

    # Second cycle
    start = l1
    cycle2 = list(range(start, start + l2))
    for i in range(l2):
        G.add_edge(cycle2[i], cycle2[(i + 1) % l2])

    # Bridge between the two cycles
    G.add_edge(cycle1[0], cycle2[0])

    return G


def generate_bicyclic_graph(n: int, rng: np.random.Generator = None) -> nx.Graph:
    """
    Generate one random bicyclic graph of order n.

    Core types:
    - vertex-sharing
    - edge-sharing
    - bridge-connected

    Then attach remaining vertices one by one to existing vertices.
    """
    if rng is None:
        rng = np.random.default_rng()

    if n < 4:
        raise ValueError("Bicyclic graph requires n >= 4.")

    core_types = ["vertex-sharing", "edge-sharing", "bridge-connected"]

    max_attempts = 200
    for _ in range(max_attempts):
        core_type = str(rng.choice(core_types))

        # Sample admissible cycle lengths depending on core type
        if core_type == "vertex-sharing":
            # core size = l1 + l2 - 1 <= n
            possible = [(l1, l2) for l1 in range(3, n + 1)
                        for l2 in range(3, n + 1)
                        if (l1 + l2 - 1) <= n]
            l1, l2 = possible[int(rng.integers(len(possible)))]
            G = make_vertex_sharing_core(l1, l2)

        elif core_type == "edge-sharing":
            # core size = l1 + l2 - 2 <= n
            possible = [(l1, l2) for l1 in range(3, n + 1)
                        for l2 in range(3, n + 1)
                        if (l1 + l2 - 2) <= n]
            l1, l2 = possible[int(rng.integers(len(possible)))]
            G = make_edge_sharing_core(l1, l2)

        else:  # bridge-connected
            # core size = l1 + l2 <= n
            possible = [(l1, l2) for l1 in range(3, n + 1)
                        for l2 in range(3, n + 1)
                        if (l1 + l2) <= n]
            l1, l2 = possible[int(rng.integers(len(possible)))]
            G = make_bridge_connected_core(l1, l2)

        # Attach remaining vertices as pendant-tree extensions
        next_node = max(G.nodes()) + 1 if G.number_of_nodes() > 0 else 0
        while G.number_of_nodes() < n:
            attach_to = int(rng.choice(list(G.nodes())))
            G.add_node(next_node)
            G.add_edge(next_node, attach_to)
            next_node += 1

        # Validate
        if is_bicyclic_graph(G):
            return G

    raise RuntimeError("Failed to generate a valid bicyclic graph after many attempts.")

In [ ]:
# Quick sanity test for bicyclic generation
rng = np.random.default_rng(42)

for n in [6, 10, 15]:
    G = generate_bicyclic_graph(n, rng)
    print(
        f"n={n}, nodes={G.number_of_nodes()}, edges={G.number_of_edges()}, "
        f"mu={cyclomatic_number(G)}, bicyclic={is_bicyclic_graph(G)}"
    )

In [ ]:
G = generate_bicyclic_graph(10, np.random.default_rng(123))
plt.figure(figsize=(4, 4))
nx.draw(G, with_labels=True, node_size=500)
plt.show()

In [ ]:
# ==========================================
# STEP 6: Graph hashing and duplicate control
# ==========================================

from networkx.algorithms.graph_hashing import weisfeiler_lehman_graph_hash

def graph_hash_key(G: nx.Graph) -> str:
    """
    Returns an isomorphism-invariant hash for an unlabeled simple graph.
    Used for duplicate removal in the synthetic dataset pipeline.
    """
    return weisfeiler_lehman_graph_hash(G)


def is_new_graph(G: nx.Graph, seen_hashes: set) -> bool:
    """
    Check whether graph G is new w.r.t. previously seen graph hashes.
    """
    return graph_hash_key(G) not in seen_hashes


def register_graph(G: nx.Graph, seen_hashes: set) -> str:
    """
    Register graph G into the seen-hash set and return its hash key.
    """
    h = graph_hash_key(G)
    seen_hashes.add(h)
    return h

In [ ]:
# Sanity test for duplicate detection

seen_hashes = set()

G1 = nx.cycle_graph(6)
G2 = nx.relabel_nodes(nx.cycle_graph(6), {i: (i + 10) for i in range(6)})  # isomorphic to G1
G3 = nx.path_graph(6)

print("G1 new:", is_new_graph(G1, seen_hashes))
register_graph(G1, seen_hashes)

print("G2 new (should be False):", is_new_graph(G2, seen_hashes))
print("G3 new (should be True):", is_new_graph(G3, seen_hashes))

In [ ]:
# ==========================================
# STEP 7: Structure-preserving transformations
# ==========================================

def get_leaf_nodes(G: nx.Graph):
    return [v for v in G.nodes() if G.degree(v) == 1]


def get_cycle_nodes_unicyclic(G: nx.Graph):
    """
    For a unicyclic graph, return the node set of the unique cycle.
    """
    cycles = nx.cycle_basis(G)
    if len(cycles) != 1:
        return set()
    return set(cycles[0])


def get_cycle_nodes_bicyclic(G: nx.Graph):
    """
    For a bicyclic graph, return the union of nodes appearing in the cycle basis.
    """
    cycles = nx.cycle_basis(G)
    core_nodes = set()
    for c in cycles:
        core_nodes.update(c)
    return core_nodes


def get_pendant_subtree_nodes(G: nx.Graph, leaf: int, attach: int):
    """
    Given a leaf and its attachment edge (leaf, attach), return the subtree
    detached on the leaf side when removing that edge.
    """
    H = G.copy()
    H.remove_edge(leaf, attach)
    components = list(nx.connected_components(H))
    for comp in components:
        if leaf in comp:
            return set(comp)
    return {leaf}


def get_non_core_components_attached_to_core(G: nx.Graph, core_nodes: set):
    """
    Return a list of tuples:
        (core_attach_node, component_root, component_nodes)
    for non-core connected components attached to the bicyclic core.
    """
    H = G.copy()
    H.remove_nodes_from(core_nodes)

    components_info = []
    for comp in nx.connected_components(H):
        comp = set(comp)

        core_neighbors = set()
        boundary_nodes = set()

        for u in comp:
            for v in G.neighbors(u):
                if v in core_nodes:
                    core_neighbors.add(v)
                    boundary_nodes.add(u)

        # only keep pendant-like components attached to exactly one core node
        if len(core_neighbors) == 1 and len(boundary_nodes) >= 1:
            core_attach = next(iter(core_neighbors))
            root = next(iter(boundary_nodes))
            components_info.append((core_attach, root, comp))

    return components_info


def pendant_edge_relocation_unicyclic(G: nx.Graph, rng=None):
    """
    Move a leaf attached to one cycle node to another cycle node.
    """
    if rng is None:
        rng = np.random.default_rng()

    if not is_unicyclic_graph(G):
        return None

    cycle_nodes = list(get_cycle_nodes_unicyclic(G))
    candidate_leaves = []

    for y in get_leaf_nodes(G):
        nbr = next(iter(G.neighbors(y)))
        if nbr in cycle_nodes:
            candidate_leaves.append((y, nbr))

    if not candidate_leaves:
        return None

    y, x = candidate_leaves[int(rng.integers(len(candidate_leaves)))]
    possible_targets = [z for z in cycle_nodes if z != x and not G.has_edge(y, z)]
    if not possible_targets:
        return None

    z = possible_targets[int(rng.integers(len(possible_targets)))]
    H = G.copy()
    H.remove_edge(x, y)
    H.add_edge(z, y)

    return H if is_unicyclic_graph(H) else None


def pendant_subtree_reattachment_unicyclic(G: nx.Graph, rng=None):
    """
    Move a pendant subtree from one cycle vertex to another cycle vertex.
    """
    if rng is None:
        rng = np.random.default_rng()

    if not is_unicyclic_graph(G):
        return None

    cycle_nodes = list(get_cycle_nodes_unicyclic(G))
    candidate_roots = []

    for x in cycle_nodes:
        for nbr in G.neighbors(x):
            if nbr not in cycle_nodes:
                candidate_roots.append((x, nbr))

    if not candidate_roots:
        return None

    x, root = candidate_roots[int(rng.integers(len(candidate_roots)))]
    possible_targets = [z for z in cycle_nodes if z != x and not G.has_edge(root, z)]
    if not possible_targets:
        return None

    z = possible_targets[int(rng.integers(len(possible_targets)))]
    H = G.copy()
    H.remove_edge(x, root)
    H.add_edge(z, root)

    return H if is_unicyclic_graph(H) else None


def pendant_relocation_bicyclic(G: nx.Graph, rng=None):
    """
    Reattach a non-core pendant component from one bicyclic core vertex
    to another admissible core vertex.
    """
    if rng is None:
        rng = np.random.default_rng()

    if not is_bicyclic_graph(G):
        return None

    core_nodes = get_cycle_nodes_bicyclic(G)
    attached_components = get_non_core_components_attached_to_core(G, core_nodes)

    if not attached_components:
        return None

    x, root, comp_nodes = attached_components[int(rng.integers(len(attached_components)))]
    possible_targets = [z for z in core_nodes if z != x and not G.has_edge(root, z)]
    if not possible_targets:
        return None

    z = possible_targets[int(rng.integers(len(possible_targets)))]
    H = G.copy()
    H.remove_edge(x, root)
    H.add_edge(z, root)

    return H if is_bicyclic_graph(H) else None


def core_preserving_rewiring_bicyclic(G: nx.Graph, rng=None):
    """
    Conservative bicyclic rewiring:
    choose a leaf or subtree root outside the bicyclic core and reconnect it
    to another admissible core node while preserving simplicity, connectivity,
    and cyclomatic number 2.
    """
    if rng is None:
        rng = np.random.default_rng()

    if not is_bicyclic_graph(G):
        return None

    core_nodes = get_cycle_nodes_bicyclic(G)
    candidates = []

    for x, root, comp_nodes in get_non_core_components_attached_to_core(G, core_nodes):
        candidates.append((x, root))

    if not candidates:
        return None

    x, root = candidates[int(rng.integers(len(candidates)))]
    possible_targets = [z for z in core_nodes if z != x and not G.has_edge(root, z)]
    if not possible_targets:
        return None

    z = possible_targets[int(rng.integers(len(possible_targets)))]
    H = G.copy()
    H.remove_edge(x, root)
    H.add_edge(z, root)

    return H if is_bicyclic_graph(H) else None


def generate_valid_transformations(G: nx.Graph, max_transforms: int = 2, rng=None):
    """
    Try to generate up to max_transforms valid transformed graphs for G.
    Returns a list of unique valid transformed graphs.
    """
    if rng is None:
        rng = np.random.default_rng()

    transforms = []
    seen_local = set()
    original_hash = graph_hash_key(G)

    if is_unicyclic_graph(G):
        ops = [
            pendant_edge_relocation_unicyclic,
            pendant_subtree_reattachment_unicyclic,
        ]
    elif is_bicyclic_graph(G):
        ops = [
            pendant_relocation_bicyclic,
            core_preserving_rewiring_bicyclic,
        ]
    else:
        return transforms

    attempts = 0
    max_attempts = 50 * max_transforms

    while len(transforms) < max_transforms and attempts < max_attempts:
        op = ops[int(rng.integers(len(ops)))]
        H = op(G, rng)
        attempts += 1

        if H is None:
            continue

        h = graph_hash_key(H)
        if h == original_hash or h in seen_local:
            continue

        seen_local.add(h)
        transforms.append(H)

    return transforms

In [ ]:
rng = np.random.default_rng(42)

# Unicyclic test
G_u = generate_unicyclic_graph(10, rng)
trans_u = generate_valid_transformations(G_u, max_transforms=2, rng=rng)
print("Unicyclic original mu:", cyclomatic_number(G_u), "| transformed:", [cyclomatic_number(H) for H in trans_u])
print("Unicyclic valid:", [is_unicyclic_graph(H) for H in trans_u])

# Bicyclic test
G_b = generate_bicyclic_graph(10, rng)
trans_b = generate_valid_transformations(G_b, max_transforms=2, rng=rng)
print("Bicyclic original mu:", cyclomatic_number(G_b), "| transformed:", [cyclomatic_number(H) for H in trans_b])
print("Bicyclic valid:", [is_bicyclic_graph(H) for H in trans_b])
print("Number of bicyclic transforms:", len(trans_b))

Unicyclic original mu: 1 | transformed: [1, 1]
Unicyclic valid: [True, True]
Bicyclic original mu: 2 | transformed: []
Bicyclic valid: []
Number of bicyclic transforms: 0


In [ ]:
core_nodes = get_cycle_nodes_bicyclic(G_b)
attached_components = get_non_core_components_attached_to_core(G_b, core_nodes)

print("Core nodes:", sorted(core_nodes))
print("Number of attached non-core components:", len(attached_components))
print("Leaf nodes:", get_leaf_nodes(G_b))
print("Nodes:", G_b.number_of_nodes(), "Edges:", G_b.number_of_edges(), "mu:", cyclomatic_number(G_b))

for i, item in enumerate(attached_components):
    x, root, comp = item
    print(f"Component {i}: attached_to_core={x}, root={root}, size={len(comp)}, nodes={sorted(comp)}")

In [ ]:
# Visualize one original and one transformed example
if trans_u:
    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    nx.draw(G_u, with_labels=True, node_size=400)
    plt.title("Original unicyclic")

    plt.subplot(1, 2, 2)
    nx.draw(trans_u[0], with_labels=True, node_size=400)
    plt.title("Transformed unicyclic")
    plt.show()

In [ ]:
# ==========================================
# STEP 8: Synthetic dataset generation
# ==========================================

def build_synthetic_graph_dataset(
    n_min=N_MIN,
    n_max=N_MAX,
    num_unicyclic_per_order=NUM_UNICYCLIC_PER_ORDER,
    num_bicyclic_per_order=NUM_BICYCLIC_PER_ORDER,
    max_transforms_per_graph=MAX_TRANSFORMS_PER_GRAPH,
    seed=SEED
):
    """
    Build synthetic graph dataset:
    - generate unique unicyclic and bicyclic graphs
    - apply up to T valid transformations per retained graph
    - remove duplicates via graph hash
    - store metadata for later splitting and analysis
    """
    rng = np.random.default_rng(seed)

    dataset = []
    seen_hashes = set()

    stats = {
        "original_unicyclic": 0,
        "original_bicyclic": 0,
        "transformed_unicyclic": 0,
        "transformed_bicyclic": 0,
        "duplicates_skipped": 0,
        "transform_duplicates_skipped": 0,
    }

    graph_id = 0

    for n in range(n_min, n_max + 1):
        # -------------------------
        # Unicyclic graphs
        # -------------------------
        for _ in range(num_unicyclic_per_order):
            G = generate_unicyclic_graph(n, rng)
            h = graph_hash_key(G)

            if h in seen_hashes:
                stats["duplicates_skipped"] += 1
                continue

            seen_hashes.add(h)
            dataset.append({
                "graph_id": graph_id,
                "graph": G,
                "family": "unicyclic",
                "order": n,
                "is_transformed": 0,
                "parent_graph_id": None,
                "graph_hash": h,
            })
            stats["original_unicyclic"] += 1
            parent_id = graph_id
            graph_id += 1

            # up to T valid transforms
            transformed_graphs = generate_valid_transformations(
                G, max_transforms=max_transforms_per_graph, rng=rng
            )

            for H in transformed_graphs:
                hh = graph_hash_key(H)
                if hh in seen_hashes:
                    stats["transform_duplicates_skipped"] += 1
                    continue

                seen_hashes.add(hh)
                dataset.append({
                    "graph_id": graph_id,
                    "graph": H,
                    "family": "unicyclic",
                    "order": H.number_of_nodes(),
                    "is_transformed": 1,
                    "parent_graph_id": parent_id,
                    "graph_hash": hh,
                })
                stats["transformed_unicyclic"] += 1
                graph_id += 1

        # -------------------------
        # Bicyclic graphs
        # -------------------------
        for _ in range(num_bicyclic_per_order):
            G = generate_bicyclic_graph(n, rng)
            h = graph_hash_key(G)

            if h in seen_hashes:
                stats["duplicates_skipped"] += 1
                continue

            seen_hashes.add(h)
            dataset.append({
                "graph_id": graph_id,
                "graph": G,
                "family": "bicyclic",
                "order": n,
                "is_transformed": 0,
                "parent_graph_id": None,
                "graph_hash": h,
            })
            stats["original_bicyclic"] += 1
            parent_id = graph_id
            graph_id += 1

            # up to T valid transforms
            transformed_graphs = generate_valid_transformations(
                G, max_transforms=max_transforms_per_graph, rng=rng
            )

            for H in transformed_graphs:
                hh = graph_hash_key(H)
                if hh in seen_hashes:
                    stats["transform_duplicates_skipped"] += 1
                    continue

                seen_hashes.add(hh)
                dataset.append({
                    "graph_id": graph_id,
                    "graph": H,
                    "family": "bicyclic",
                    "order": H.number_of_nodes(),
                    "is_transformed": 1,
                    "parent_graph_id": parent_id,
                    "graph_hash": hh,
                })
                stats["transformed_bicyclic"] += 1
                graph_id += 1

    df_meta = pd.DataFrame([
        {
            "graph_id": item["graph_id"],
            "family": item["family"],
            "order": item["order"],
            "is_transformed": item["is_transformed"],
            "parent_graph_id": item["parent_graph_id"],
            "graph_hash": item["graph_hash"],
        }
        for item in dataset
    ])

    return dataset, df_meta, stats

In [ ]:
dataset, df_meta, stats = build_synthetic_graph_dataset()

print("Dataset size:", len(dataset))
print("\nStats:")
for k, v in stats.items():
    print(f"{k}: {v}")

print("\nFamily counts:")
print(df_meta["family"].value_counts())

print("\nTransformed counts:")
print(df_meta["is_transformed"].value_counts())

print("\nHead:")
print(df_meta.head())

In [ ]:
print("Unique hashes:", df_meta["graph_hash"].nunique())
print("Total rows   :", len(df_meta))
assert df_meta["graph_hash"].nunique() == len(df_meta), "Duplicate graphs still exist!"
print("Duplicate check passed.")

In [ ]:
# ==========================================
# STEP 9: Exact computation of topological indices
# ==========================================

def canonical_graph_signature(G: nx.Graph):
    """
    Canonical immutable signature for memoization on unlabeled simple graphs.
    We relabel nodes to consecutive integers in sorted order and store edges.
    """
    nodes = sorted(G.nodes())
    mapping = {u: i for i, u in enumerate(nodes)}
    edges = sorted((min(mapping[u], mapping[v]), max(mapping[u], mapping[v])) for u, v in G.edges())
    return (len(nodes), tuple(edges))


# -------------------------------------------------
# Wiener index
# -------------------------------------------------
def wiener_index(G: nx.Graph) -> int:
    lengths = dict(nx.all_pairs_shortest_path_length(G))
    total = 0
    nodes = list(G.nodes())
    for i, u in enumerate(nodes):
        for v in nodes[i+1:]:
            total += lengths[u][v]
    return total


# -------------------------------------------------
# First Zagreb index
# -------------------------------------------------
def first_zagreb_index(G: nx.Graph) -> int:
    return sum(G.degree(v) ** 2 for v in G.nodes())


# -------------------------------------------------
# Second Zagreb index
# -------------------------------------------------
def second_zagreb_index(G: nx.Graph) -> int:
    return sum(G.degree(u) * G.degree(v) for u, v in G.edges())


# -------------------------------------------------
# Randic index
# -------------------------------------------------
def randic_index(G: nx.Graph) -> float:
    return sum(1.0 / math.sqrt(G.degree(u) * G.degree(v)) for u, v in G.edges())


# -------------------------------------------------
# Merrifield-Simmons index
# Number of independent sets
# -------------------------------------------------
_ms_cache = {}

def merrifield_simmons_index(G: nx.Graph) -> int:
    sig = canonical_graph_signature(G)
    if sig in _ms_cache:
        return _ms_cache[sig]

    n = G.number_of_nodes()
    if n == 0:
        return 1
    if n == 1:
        return 2  # {} and {v}

    v = next(iter(G.nodes()))

    # Exclude v
    G_excl = G.copy()
    G_excl.remove_node(v)

    # Include v -> remove v and all its neighbors
    G_incl = G.copy()
    nbrs = list(G.neighbors(v))
    G_incl.remove_nodes_from([v] + nbrs)

    result = merrifield_simmons_index(G_excl) + merrifield_simmons_index(G_incl)
    _ms_cache[sig] = result
    return result


# -------------------------------------------------
# Hosoya index
# Number of matchings
# -------------------------------------------------
_z_cache = {}

def hosoya_index(G: nx.Graph) -> int:
    sig = canonical_graph_signature(G)
    if sig in _z_cache:
        return _z_cache[sig]

    m = G.number_of_edges()
    if m == 0:
        return 1  # only empty matching

    u, v = next(iter(G.edges()))

    # Exclude edge (u,v)
    G_excl = G.copy()
    G_excl.remove_edge(u, v)

    # Include edge (u,v) -> remove both endpoints
    G_incl = G.copy()
    G_incl.remove_nodes_from([u, v])

    result = hosoya_index(G_excl) + hosoya_index(G_incl)
    _z_cache[sig] = result
    return result


# -------------------------------------------------
# Full exact target vector
# -------------------------------------------------
def compute_exact_target_vector(G: nx.Graph):
    """
    Returns target vector in the paper order:
    [W, MS, Z, M1, M2, R]
    """
    W = wiener_index(G)
    MS = merrifield_simmons_index(G)
    Z = hosoya_index(G)
    M1 = first_zagreb_index(G)
    M2 = second_zagreb_index(G)
    R = randic_index(G)

    return np.array([W, MS, Z, M1, M2, R], dtype=float)

In [ ]:
# Quick sanity checks on small graphs

G1 = nx.path_graph(4)
print("Path graph P4")
print("W  =", wiener_index(G1))
print("MS =", merrifield_simmons_index(G1))
print("Z  =", hosoya_index(G1))
print("M1 =", first_zagreb_index(G1))
print("M2 =", second_zagreb_index(G1))
print("R  =", randic_index(G1))
print("y  =", compute_exact_target_vector(G1))

print("\nUnicyclic sample")
G2 = dataset[0]["graph"]
print("family:", dataset[0]["family"], "order:", G2.number_of_nodes(), "mu:", cyclomatic_number(G2))
print("y =", compute_exact_target_vector(G2))

In [ ]:
# ==========================================
# STEP 10: Label the dataset with exact targets
# ==========================================

def label_graph_dataset(dataset):
    """
    Compute exact target vector for every graph in dataset.
    Returns:
        - updated dataset with 'target'
        - dataframe with metadata + targets
    """
    start_time = time.time()

    labeled_rows = []
    total = len(dataset)

    for i, item in enumerate(dataset):
        G = item["graph"]
        y = compute_exact_target_vector(G)

        item["target"] = y

        labeled_rows.append({
            "graph_id": item["graph_id"],
            "family": item["family"],
            "order": item["order"],
            "is_transformed": item["is_transformed"],
            "parent_graph_id": item["parent_graph_id"],
            "graph_hash": item["graph_hash"],
            "W": y[0],
            "MS": y[1],
            "Z": y[2],
            "M1": y[3],
            "M2": y[4],
            "R": y[5],
        })

        # progress
        if (i + 1) % 500 == 0 or (i + 1) == total:
            elapsed = time.time() - start_time
            print(f"Labeled {i+1}/{total} graphs | elapsed: {elapsed:.2f} sec")

    df_targets = pd.DataFrame(labeled_rows)

    total_elapsed = time.time() - start_time
    print(f"\nTotal labeling time: {total_elapsed:.2f} sec")
    print(f"Average labeling time per graph: {1000 * total_elapsed / total:.4f} ms")

    return dataset, df_targets

In [ ]:
dataset, df_targets = label_graph_dataset(dataset)

In [ ]:
print(df_targets.head())
print("\nShape:", df_targets.shape)
print("\nMissing values:")
print(df_targets.isnull().sum())

In [ ]:
# ==========================================
# STEP 11: Descriptive statistics of targets
# ==========================================

target_cols = ["W", "MS", "Z", "M1", "M2", "R"]

target_stats = df_targets[target_cols].agg(["min", "max", "mean", "std"]).T
target_stats = target_stats.rename(columns={
    "min": "Minimum",
    "max": "Maximum",
    "mean": "Mean",
    "std": "Standard Deviation"
})

print(target_stats)

In [ ]:
target_stats_rounded = target_stats.copy()
target_stats_rounded["Minimum"] = target_stats_rounded["Minimum"].round(2)
target_stats_rounded["Maximum"] = target_stats_rounded["Maximum"].round(2)
target_stats_rounded["Mean"] = target_stats_rounded["Mean"].round(2)
target_stats_rounded["Standard Deviation"] = target_stats_rounded["Standard Deviation"].round(2)

print(target_stats_rounded)

In [ ]:
for idx, row in target_stats_rounded.iterrows():
    print(f"{idx} & {row['Minimum']} & {row['Maximum']} & {row['Mean']} & {row['Standard Deviation']} \\\\")

In [ ]:
# ==========================================
# STEP 12: Stratified train/validation/test split
# ==========================================

df_all = df_targets.copy()

# Joint stratification by graph family and graph order
df_all["stratify_label"] = df_all["family"].astype(str) + "_" + df_all["order"].astype(str)

# First split: train vs temp (val+test)
df_train, df_temp = train_test_split(
    df_all,
    test_size=(1.0 - TRAIN_RATIO),
    random_state=SEED,
    stratify=df_all["stratify_label"]
)

# Second split: val vs test
# temp = 30%, so split equally into 15% and 15%
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.5,
    random_state=SEED,
    stratify=df_temp["stratify_label"]
)

print("Train size:", len(df_train))
print("Val size  :", len(df_val))
print("Test size :", len(df_test))
print("Total     :", len(df_all))

In [ ]:
print("\nSplit ratios:")
print("Train:", len(df_train) / len(df_all))
print("Val  :", len(df_val) / len(df_all))
print("Test :", len(df_test) / len(df_all))

In [ ]:
def summarize_split(df, name):
    print(f"\n{name} family counts:")
    print(df["family"].value_counts().sort_index())

    print(f"\n{name} order counts:")
    print(df["order"].value_counts().sort_index().head())
    print("...")
    print(df["order"].value_counts().sort_index().tail())

summarize_split(df_train, "Train")
summarize_split(df_val, "Validation")
summarize_split(df_test, "Test")

In [ ]:
graph_lookup = {item["graph_id"]: item for item in dataset}
print("Lookup size:", len(graph_lookup))

In [ ]:
# ==========================================
# STEP 13: Fit target and feature scalers
# ==========================================

TARGET_COLS = ["W", "MS", "Z", "M1", "M2", "R"]

# ------------------------------------------
# 13A. Target scaler (fit on training only)
# ------------------------------------------
target_scaler = StandardScaler()
target_scaler.fit(df_train[TARGET_COLS].values)

print("Target scaler mean:")
print(target_scaler.mean_)

print("\nTarget scaler std:")
print(target_scaler.scale_)

In [ ]:
# ==========================================
# STEP 13B: Node feature extraction helpers
# ==========================================

def get_cycle_membership_vector(G: nx.Graph):
    """
    Returns dict: node -> 1 if node belongs to at least one cycle, else 0
    """
    cycle_nodes = set()
    for cyc in nx.cycle_basis(G):
        cycle_nodes.update(cyc)
    return {v: int(v in cycle_nodes) for v in G.nodes()}


def get_pendant_neighbor_count(G: nx.Graph):
    """
    Returns dict: node -> number of leaf neighbors
    """
    leaf_set = set(get_leaf_nodes(G))
    counts = {}
    for v in G.nodes():
        counts[v] = sum((u in leaf_set) for u in G.neighbors(v))
    return counts


def get_local_eccentricity(G: nx.Graph):
    """
    Returns dict: node -> eccentricity(v)
    """
    return nx.eccentricity(G)


def extract_node_feature_matrix(G: nx.Graph):
    """
    Node features in paper order:
    1. normalized degree
    2. cycle-membership indicator
    3. number of pendant neighbors
    4. leaf indicator
    5. local eccentricity normalized by graph order
    """
    nodes = sorted(G.nodes())
    n = G.number_of_nodes()

    cycle_membership = get_cycle_membership_vector(G)
    pendant_neighbor_count = get_pendant_neighbor_count(G)
    eccentricity = get_local_eccentricity(G)

    X = []
    for v in nodes:
        deg_norm = G.degree(v) / max(1, n - 1)
        cyc_ind = float(cycle_membership[v])
        pend_nbrs = float(pendant_neighbor_count[v])
        leaf_ind = float(G.degree(v) == 1)
        ecc_norm = float(eccentricity[v]) / max(1, n)

        X.append([deg_norm, cyc_ind, pend_nbrs, leaf_ind, ecc_norm])

    return np.array(X, dtype=np.float32), nodes

In [ ]:
# ==========================================
# STEP 13C: Fit node-feature scaler on training graphs only
# ==========================================

all_train_node_features = []

for _, row in df_train.iterrows():
    graph_id = row["graph_id"]
    G = graph_lookup[graph_id]["graph"]
    X, _ = extract_node_feature_matrix(G)
    all_train_node_features.append(X)

all_train_node_features = np.vstack(all_train_node_features)

feature_scaler = StandardScaler()
feature_scaler.fit(all_train_node_features)

print("Node feature scaler mean:")
print(feature_scaler.mean_)

print("\nNode feature scaler std:")
print(feature_scaler.scale_)

In [ ]:
# ==========================================
# STEP 14: Build PyG Data objects
# ==========================================

def nx_to_edge_index(G: nx.Graph, node_order):
    """
    Convert NetworkX graph to PyG edge_index using a fixed node order.
    Undirected edges are stored in both directions.
    """
    node_to_idx = {node: i for i, node in enumerate(node_order)}
    edges = []

    for u, v in G.edges():
        ui = node_to_idx[u]
        vi = node_to_idx[v]
        edges.append([ui, vi])
        edges.append([vi, ui])

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return edge_index


def build_pyg_data_object(row, graph_lookup, feature_scaler, target_scaler):
    """
    Build one PyG Data object from one dataframe row.
    """
    graph_id = row["graph_id"]
    item = graph_lookup[graph_id]
    G = item["graph"]

    # Node features
    X_raw, node_order = extract_node_feature_matrix(G)
    X_scaled = feature_scaler.transform(X_raw)

    # Edge index
    edge_index = nx_to_edge_index(G, node_order)

    # Targets
    y_raw = np.array([[row["W"], row["MS"], row["Z"], row["M1"], row["M2"], row["R"]]], dtype=np.float32)
    y_scaled = target_scaler.transform(y_raw).astype(np.float32)

    data = Data(
        x=torch.tensor(X_scaled, dtype=torch.float),
        edge_index=edge_index,
        y=torch.tensor(y_scaled, dtype=torch.float)   # shape: [1, 6]
    )

    # Metadata
    data.graph_id = int(row["graph_id"])
    data.family = row["family"]
    data.order = int(row["order"])
    data.is_transformed = int(row["is_transformed"])

    return data

In [ ]:
# Build train / val / test PyG datasets
train_data_list = [
    build_pyg_data_object(row, graph_lookup, feature_scaler, target_scaler)
    for _, row in df_train.iterrows()
]

val_data_list = [
    build_pyg_data_object(row, graph_lookup, feature_scaler, target_scaler)
    for _, row in df_val.iterrows()
]

test_data_list = [
    build_pyg_data_object(row, graph_lookup, feature_scaler, target_scaler)
    for _, row in df_test.iterrows()
]

print("Train graphs:", len(train_data_list))
print("Val graphs  :", len(val_data_list))
print("Test graphs :", len(test_data_list))

In [ ]:
sample = train_data_list[0]
print(sample)
print("x shape       :", sample.x.shape)
print("edge_index    :", sample.edge_index.shape)
print("y shape       :", sample.y.shape)
print("graph_id      :", sample.graph_id)
print("family        :", sample.family)
print("order         :", sample.order)
print("is_transformed:", sample.is_transformed)

In [ ]:
train_loader = DataLoader(train_data_list, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data_list, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_data_list, batch_size=BATCH_SIZE, shuffle=False)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Test batches :", len(test_loader))

In [ ]:
# ==========================================
# STEP 15: Primary GIN model
# ==========================================

class GINRegressor(nn.Module):
    def __init__(self, in_channels, hidden_dim=64, out_dim=6, dropout=0.2):
        super().__init__()

        self.dropout = dropout

        # GIN layer 1
        mlp1 = nn.Sequential(
            nn.Linear(in_channels, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.conv1 = GINConv(mlp1)
        self.bn1 = nn.BatchNorm1d(hidden_dim)

        # GIN layer 2
        mlp2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.conv2 = GINConv(mlp2)
        self.bn2 = nn.BatchNorm1d(hidden_dim)

        # GIN layer 3
        mlp3 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.conv3 = GINConv(mlp3)
        self.bn3 = nn.BatchNorm1d(hidden_dim)

        # Two-layer regression head
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv3(x, edge_index)
        x = self.bn3(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        # Global sum pooling
        x = global_add_pool(x, batch)

        # Output head
        out = self.head(x)
        return out

In [ ]:
gin_model = GINRegressor(
    in_channels=NUM_NODE_FEATURES,
    hidden_dim=HIDDEN_DIM,
    out_dim=NUM_TARGETS,
    dropout=DROPOUT
).to(DEVICE)

print(gin_model)

In [ ]:
batch = next(iter(train_loader)).to(DEVICE)
with torch.no_grad():
    out = gin_model(batch)

print("Batch input x shape :", batch.x.shape)
print("Batch y shape       :", batch.y.shape)
print("Model output shape  :", out.shape)

In [ ]:
# ==========================================
# STEP 16: Training and validation loops
# ==========================================

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    total_graphs = 0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        pred = model(batch)
        loss = F.mse_loss(pred, batch.y)

        loss.backward()
        optimizer.step()

        num_graphs = batch.num_graphs
        total_loss += loss.item() * num_graphs
        total_graphs += num_graphs

    return total_loss / max(total_graphs, 1)


@torch.no_grad()
def evaluate_loss(model, loader, device):
    model.eval()
    total_loss = 0.0
    total_graphs = 0

    for batch in loader:
        batch = batch.to(device)
        pred = model(batch)
        loss = F.mse_loss(pred, batch.y)

        num_graphs = batch.num_graphs
        total_loss += loss.item() * num_graphs
        total_graphs += num_graphs

    return total_loss / max(total_graphs, 1)


def train_gin_model(
    train_loader,
    val_loader,
    in_channels=NUM_NODE_FEATURES,
    hidden_dim=HIDDEN_DIM,
    out_dim=NUM_TARGETS,
    dropout=DROPOUT,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    max_epochs=MAX_EPOCHS,
    patience=EARLY_STOPPING_PATIENCE,
    device=DEVICE,
):
    model = GINRegressor(
        in_channels=in_channels,
        hidden_dim=hidden_dim,
        out_dim=out_dim,
        dropout=dropout,
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    history = {
        "train_loss": [],
        "val_loss": [],
    }

    best_val_loss = float("inf")
    best_state = None
    best_epoch = -1
    patience_counter = 0

    start_time = time.time()

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_loss = evaluate_loss(model, val_loader, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if epoch % 10 == 0 or epoch == 1:
            print(
                f"Epoch {epoch:03d} | "
                f"Train Loss: {train_loss:.6f} | "
                f"Val Loss: {val_loss:.6f}"
            )

        if patience_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch}.")
            break

    total_training_time = time.time() - start_time

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"\nBest epoch: {best_epoch}")
    print(f"Best val loss: {best_val_loss:.6f}")
    print(f"Training time: {total_training_time:.2f} sec")

    return model, history, best_val_loss, best_epoch, total_training_time

In [ ]:
gin_model, gin_history, gin_best_val_loss, gin_best_epoch, gin_train_time = train_gin_model(
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE
)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(gin_history["train_loss"], label="Train Loss")
plt.plot(gin_history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("GIN Training Curve")
plt.legend()
plt.grid(True, linestyle="--", linewidth=0.6, alpha=0.7)
plt.show()

In [ ]:
print("Best epoch      :", gin_best_epoch)
print("Best val loss   :", gin_best_val_loss)
print("Training time(s):", gin_train_time)
print("Training time(m):", gin_train_time / 60.0)

In [ ]:
# ==========================================
# STEP 17: Test-set evaluation for GIN
# ==========================================

@torch.no_grad()
def collect_predictions(model, loader, device):
    model.eval()

    y_true_std = []
    y_pred_std = []
    meta_rows = []

    for batch in loader:
        batch = batch.to(device)
        pred = model(batch)

        y_true_std.append(batch.y.cpu().numpy())
        y_pred_std.append(pred.cpu().numpy())

        # metadata
        graph_ids = batch.graph_id
        families = batch.family
        orders = batch.order
        transformed_flags = batch.is_transformed

        for i in range(batch.num_graphs):
            meta_rows.append({
                "graph_id": int(graph_ids[i]),
                "family": families[i],
                "order": int(orders[i]),
                "is_transformed": int(transformed_flags[i]),
            })

    y_true_std = np.vstack(y_true_std)
    y_pred_std = np.vstack(y_pred_std)
    df_meta_pred = pd.DataFrame(meta_rows)

    return y_true_std, y_pred_std, df_meta_pred


def rmse_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def mape_score(y_true, y_pred, eps=1e-8):
    return np.mean(np.abs((y_true - y_pred) / (y_true + eps))) * 100.0


def compute_metrics_table(y_true, y_pred, target_names):
    rows = []
    for j, name in enumerate(target_names):
        yt = y_true[:, j]
        yp = y_pred[:, j]

        rows.append({
            "Index": name,
            "MAE": mean_absolute_error(yt, yp),
            "RMSE": rmse_score(yt, yp),
            "R2": r2_score(yt, yp),
            "MAPE (%)": mape_score(yt, yp),
        })

    df_metrics = pd.DataFrame(rows)

    macro_row = {
        "Index": "Macro average",
        "MAE": df_metrics["MAE"].mean(),
        "RMSE": df_metrics["RMSE"].mean(),
        "R2": df_metrics["R2"].mean(),
        "MAPE (%)": df_metrics["MAPE (%)"].mean(),
    }
    df_metrics = pd.concat([df_metrics, pd.DataFrame([macro_row])], ignore_index=True)
    return df_metrics

In [ ]:
y_true_std, y_pred_std, df_test_meta_pred = collect_predictions(gin_model, test_loader, DEVICE)

print("Standardized shapes:")
print("y_true_std:", y_true_std.shape)
print("y_pred_std:", y_pred_std.shape)
print(df_test_meta_pred.head())

In [ ]:
y_true_orig = target_scaler.inverse_transform(y_true_std)
y_pred_orig = target_scaler.inverse_transform(y_pred_std)

print("Original-scale shapes:")
print("y_true_orig:", y_true_orig.shape)
print("y_pred_orig:", y_pred_orig.shape)

In [ ]:
df_metrics_std = compute_metrics_table(y_true_std, y_pred_std, TARGET_NAMES)
df_gin_metrics_std = df_metrics_std.copy()
print("GIN test metrics on standardized scale:")
print(df_metrics_std.round(6))

In [ ]:
df_metrics_orig = compute_metrics_table(y_true_orig, y_pred_orig, TARGET_NAMES)
df_gin_metrics_orig = df_metrics_orig.copy()
print("GIN test metrics on original scale:")
print(df_metrics_orig.round(6))

In [ ]:
# ==========================================
# STEP 18: Subgroup evaluation for GIN
# ==========================================

def compute_metrics_from_arrays(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse_score(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "MAPE (%)": mape_score(y_true, y_pred),
    }


def compute_macro_metrics(y_true, y_pred):
    rows = []
    for j in range(y_true.shape[1]):
        rows.append(compute_metrics_from_arrays(y_true[:, j], y_pred[:, j]))

    df = pd.DataFrame(rows)
    return {
        "MAE": df["MAE"].mean(),
        "RMSE": df["RMSE"].mean(),
        "R2": df["R2"].mean(),
        "MAPE (%)": df["MAPE (%)"].mean(),
    }


# Merge predictions metadata with arrays
df_eval = df_test_meta_pred.copy()

for j, name in enumerate(TARGET_NAMES):
    df_eval[f"true_{name}"] = y_true_std[:, j]
    df_eval[f"pred_{name}"] = y_pred_std[:, j]

print(df_eval.head())

In [ ]:
family_rows = []

for family_name, sub_df in df_eval.groupby("family"):
    y_true_sub = sub_df[[f"true_{t}" for t in TARGET_NAMES]].values
    y_pred_sub = sub_df[[f"pred_{t}" for t in TARGET_NAMES]].values

    m = compute_macro_metrics(y_true_sub, y_pred_sub)
    family_rows.append({
        "Graph family": family_name,
        "MAE": m["MAE"],
        "RMSE": m["RMSE"],
        "R2": m["R2"],
        "MAPE (%)": m["MAPE (%)"],
    })

df_family_perf = pd.DataFrame(family_rows)
print(df_family_perf.round(6))

In [ ]:
def order_bucket(order):
    if 6 <= order <= 10:
        return "6--10"
    elif 11 <= order <= 15:
        return "11--15"
    elif 16 <= order <= 20:
        return "16--20"
    elif 21 <= order <= 25:
        return "21--25"
    elif 26 <= order <= 30:
        return "26--30"
    else:
        return "other"

df_eval["order_bucket"] = df_eval["order"].apply(order_bucket)

order_rows = []

for bucket in ["6--10", "11--15", "16--20", "21--25", "26--30"]:
    sub_df = df_eval[df_eval["order_bucket"] == bucket]

    y_true_sub = sub_df[[f"true_{t}" for t in TARGET_NAMES]].values
    y_pred_sub = sub_df[[f"pred_{t}" for t in TARGET_NAMES]].values

    m = compute_macro_metrics(y_true_sub, y_pred_sub)
    order_rows.append({
        "Graph-order interval": bucket,
        "MAE": m["MAE"],
        "RMSE": m["RMSE"],
        "R2": m["R2"],
        "MAPE (%)": m["MAPE (%)"],
    })

df_order_perf = pd.DataFrame(order_rows)
print(df_order_perf.round(6))

In [ ]:
comb_rows = []

for bucket in ["6--10", "11--15", "16--20", "21--25", "26--30"]:
    sub_df = df_eval[df_eval["order_bucket"] == bucket]

    ms_mae = mean_absolute_error(sub_df["true_MS"].values, sub_df["pred_MS"].values)
    z_mae = mean_absolute_error(sub_df["true_Z"].values, sub_df["pred_Z"].values)

    comb_rows.append({
        "Graph-order interval": bucket,
        "MS(G) MAE": ms_mae,
        "Z(G) MAE": z_mae,
    })

df_comb_order_perf = pd.DataFrame(comb_rows)
print(df_comb_order_perf.round(6))

In [ ]:
transform_rows = []

for flag, sub_df in df_eval.groupby("is_transformed"):
    subset_name = "Original graphs" if flag == 0 else "Transformed graphs"

    y_true_sub = sub_df[[f"true_{t}" for t in TARGET_NAMES]].values
    y_pred_sub = sub_df[[f"pred_{t}" for t in TARGET_NAMES]].values

    m = compute_macro_metrics(y_true_sub, y_pred_sub)
    transform_rows.append({
        "Graph subset": subset_name,
        "MAE": m["MAE"],
        "RMSE": m["RMSE"],
        "R2": m["R2"],
        "MAPE (%)": m["MAPE (%)"],
    })

df_transform_perf = pd.DataFrame(transform_rows)
print(df_transform_perf.round(6))

In [ ]:
# ==========================================
# STEP 19A: Handcrafted descriptors for baselines
# ==========================================

def extract_graph_level_descriptors(G: nx.Graph):
    """
    Handcrafted baseline descriptor vector:
    [n, m, avg_degree, degree_variance, num_leaves, num_cycle_nodes, cyclomatic_number]
    """
    n = G.number_of_nodes()
    m = G.number_of_edges()

    degrees = np.array([G.degree(v) for v in G.nodes()], dtype=np.float32)
    avg_degree = float(degrees.mean()) if len(degrees) > 0 else 0.0
    degree_variance = float(degrees.var()) if len(degrees) > 0 else 0.0

    num_leaves = int(sum(d == 1 for d in degrees))

    cycle_nodes = set()
    for cyc in nx.cycle_basis(G):
        cycle_nodes.update(cyc)
    num_cycle_nodes = int(len(cycle_nodes))

    mu = int(cyclomatic_number(G))

    return np.array(
        [n, m, avg_degree, degree_variance, num_leaves, num_cycle_nodes, mu],
        dtype=np.float32
    )

In [ ]:
# Build baseline X/y matrices from dataframe splits

def build_baseline_matrix(df_split, graph_lookup):
    X_list = []
    Y_list = []

    for _, row in df_split.iterrows():
        graph_id = row["graph_id"]
        G = graph_lookup[graph_id]["graph"]

        x = extract_graph_level_descriptors(G)
        y = np.array([row["W"], row["MS"], row["Z"], row["M1"], row["M2"], row["R"]], dtype=np.float32)

        X_list.append(x)
        Y_list.append(y)

    X = np.vstack(X_list)
    Y = np.vstack(Y_list)
    return X, Y


X_train_base, Y_train_base = build_baseline_matrix(df_train, graph_lookup)
X_val_base, Y_val_base = build_baseline_matrix(df_val, graph_lookup)
X_test_base, Y_test_base = build_baseline_matrix(df_test, graph_lookup)

print("X_train_base:", X_train_base.shape)
print("Y_train_base:", Y_train_base.shape)
print("X_val_base  :", X_val_base.shape)
print("Y_val_base  :", Y_val_base.shape)
print("X_test_base :", X_test_base.shape)
print("Y_test_base :", Y_test_base.shape)

X_train_base: (13395, 7)
Y_train_base: (13395, 6)
X_val_base  : (2871, 7)
Y_val_base  : (2871, 6)
X_test_base : (2871, 7)
Y_test_base : (2871, 6)


In [ ]:
from sklearn.preprocessing import StandardScaler

baseline_x_scaler = StandardScaler()
baseline_x_scaler.fit(X_train_base)

baseline_y_scaler = StandardScaler()
baseline_y_scaler.fit(Y_train_base)

X_train_base_std = baseline_x_scaler.transform(X_train_base)
X_val_base_std = baseline_x_scaler.transform(X_val_base)
X_test_base_std = baseline_x_scaler.transform(X_test_base)

Y_train_base_std = baseline_y_scaler.transform(Y_train_base)
Y_val_base_std = baseline_y_scaler.transform(Y_val_base)
Y_test_base_std = baseline_y_scaler.transform(Y_test_base)

print("Baseline feature scaler mean:", baseline_x_scaler.mean_)
print("Baseline target scaler mean :", baseline_y_scaler.mean_)

Baseline feature scaler mean: [20.63889511 21.15035461  2.05605886  0.92256412  6.29063083 12.21627473
  1.5114595 ]
Baseline target scaler mean : [9.79371706e+02 4.72063526e+05 1.16402674e+05 1.05685106e+02
 1.25460918e+02 9.60583888e+00]


In [ ]:
# ==========================================
# STEP 19B: Train classical baseline models
# ==========================================

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

baseline_models = {}

# 1. Linear Regression
t0 = time.time()
lin_reg = LinearRegression()
lin_reg.fit(X_train_base_std, Y_train_base_std)
lin_time = time.time() - t0
baseline_models["Linear Regression"] = {
    "model": lin_reg,
    "train_time_sec": lin_time,
}

# 2. MLP
t0 = time.time()
mlp_reg = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    learning_rate_init=1e-3,
    max_iter=200,
    random_state=SEED,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20
)
mlp_reg.fit(X_train_base_std, Y_train_base_std)
mlp_time = time.time() - t0
baseline_models["MLP"] = {
    "model": mlp_reg,
    "train_time_sec": mlp_time,
}

# 3. Random Forest
t0 = time.time()
rf_reg = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    random_state=SEED,
    n_jobs=-1
)
rf_reg.fit(X_train_base_std, Y_train_base_std)
rf_time = time.time() - t0
baseline_models["Random Forest"] = {
    "model": rf_reg,
    "train_time_sec": rf_time,
}

for name, obj in baseline_models.items():
    print(f"{name}: training time = {obj['train_time_sec']:.2f} sec")

In [ ]:
def evaluate_sklearn_model(model, X_test_std, Y_test_std, target_names):
    Y_pred_std = model.predict(X_test_std)
    df_model_metrics_std = compute_metrics_table(Y_test_std, Y_pred_std, target_names)
    return Y_pred_std, df_model_metrics_std

baseline_test_results = {}

for name, obj in baseline_models.items():
    model = obj["model"]
    Y_pred_std, df_model_metrics_std = evaluate_sklearn_model(
        model, X_test_base_std, Y_test_base_std, TARGET_NAMES
    )
    baseline_test_results[name] = {
        "Y_pred_std": Y_pred_std,
        "metrics_std": df_model_metrics_std,
        "train_time_sec": obj["train_time_sec"],
    }

for name, result in baseline_test_results.items():
    print(f"\n{name}")
    print(result["metrics_std"].round(6))

In [ ]:
overall_rows = []

for name, result in baseline_test_results.items():
    dfm = result["metrics_std"]
    macro = dfm[dfm["Index"] == "Macro average"].iloc[0]

    overall_rows.append({
        "Model": name,
        "MAE": macro["MAE"],
        "RMSE": macro["RMSE"],
        "R2": macro["R2"],
        "MAPE (%)": macro["MAPE (%)"],
        "Train Time (s)": result["train_time_sec"],
    })

df_baseline_overall = pd.DataFrame(overall_rows)
print(df_baseline_overall.round(6))

In [ ]:
# ==========================================
# STEP 19C: Graph-based baselines (GCN, GAT)
# ==========================================

class GCNRegressor(nn.Module):
    def __init__(self, in_channels, hidden_dim=64, out_dim=6, dropout=0.2):
        super().__init__()
        self.dropout = dropout

        self.conv1 = GCNConv(in_channels, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)

        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)

        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.bn3 = nn.BatchNorm1d(hidden_dim)

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv3(x, edge_index)
        x = self.bn3(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = global_add_pool(x, batch)
        out = self.head(x)
        return out


class GATRegressor(nn.Module):
    def __init__(self, in_channels, hidden_dim=64, out_dim=6, dropout=0.2, heads=4):
        super().__init__()
        self.dropout = dropout
        self.heads = heads

        self.conv1 = GATConv(in_channels, hidden_dim // heads, heads=heads, dropout=dropout)
        self.bn1 = nn.BatchNorm1d(hidden_dim)

        self.conv2 = GATConv(hidden_dim, hidden_dim // heads, heads=heads, dropout=dropout)
        self.bn2 = nn.BatchNorm1d(hidden_dim)

        self.conv3 = GATConv(hidden_dim, hidden_dim // heads, heads=heads, dropout=dropout)
        self.bn3 = nn.BatchNorm1d(hidden_dim)

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv3(x, edge_index)
        x = self.bn3(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = global_add_pool(x, batch)
        out = self.head(x)
        return out


def train_graph_model(
    model_class,
    model_name,
    train_loader,
    val_loader,
    in_channels=NUM_NODE_FEATURES,
    hidden_dim=HIDDEN_DIM,
    out_dim=NUM_TARGETS,
    dropout=DROPOUT,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    max_epochs=MAX_EPOCHS,
    patience=EARLY_STOPPING_PATIENCE,
    device=DEVICE,
    **extra_kwargs
):
    model = model_class(
        in_channels=in_channels,
        hidden_dim=hidden_dim,
        out_dim=out_dim,
        dropout=dropout,
        **extra_kwargs
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    history = {"train_loss": [], "val_loss": []}
    best_val_loss = float("inf")
    best_state = None
    best_epoch = -1
    patience_counter = 0

    start_time = time.time()

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_loss = evaluate_loss(model, val_loader, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if epoch % 10 == 0 or epoch == 1:
            print(
                f"{model_name} | Epoch {epoch:03d} | "
                f"Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}"
            )

        if patience_counter >= patience:
            print(f"{model_name} | Early stopping at epoch {epoch}")
            break

    train_time = time.time() - start_time

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"{model_name} | Best epoch: {best_epoch}")
    print(f"{model_name} | Best val loss: {best_val_loss:.6f}")
    print(f"{model_name} | Training time: {train_time:.2f} sec")

    return model, history, best_val_loss, best_epoch, train_time

In [ ]:
gcn_model, gcn_history, gcn_best_val_loss, gcn_best_epoch, gcn_train_time = train_graph_model(
    model_class=GCNRegressor,
    model_name="GCN",
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE
)

In [ ]:
gat_model, gat_history, gat_best_val_loss, gat_best_epoch, gat_train_time = train_graph_model(
    model_class=GATRegressor,
    model_name="GAT",
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    heads=4
)

In [ ]:
# GCN
y_true_std_gcn, y_pred_std_gcn, _ = collect_predictions(gcn_model, test_loader, DEVICE)
df_gcn_metrics_std = compute_metrics_table(y_true_std_gcn, y_pred_std_gcn, TARGET_NAMES)

# GAT
y_true_std_gat, y_pred_std_gat, _ = collect_predictions(gat_model, test_loader, DEVICE)
df_gat_metrics_std = compute_metrics_table(y_true_std_gat, y_pred_std_gat, TARGET_NAMES)

print("GCN standardized test metrics:")
print(df_gcn_metrics_std.round(6))

print("\nGAT standardized test metrics:")
print(df_gat_metrics_std.round(6))

In [ ]:
overall_rows = []

# Classical baselines
for name, result in baseline_test_results.items():
    macro = result["metrics_std"][result["metrics_std"]["Index"] == "Macro average"].iloc[0]
    overall_rows.append({
        "Model": name,
        "MAE": macro["MAE"],
        "RMSE": macro["RMSE"],
        "R2": macro["R2"],
        "MAPE (%)": macro["MAPE (%)"],
        "Train Time (s)": result["train_time_sec"],
    })

# GCN
macro_gcn = df_gcn_metrics_std[df_gcn_metrics_std["Index"] == "Macro average"].iloc[0]
overall_rows.append({
    "Model": "GCN",
    "MAE": macro_gcn["MAE"],
    "RMSE": macro_gcn["RMSE"],
    "R2": macro_gcn["R2"],
    "MAPE (%)": macro_gcn["MAPE (%)"],
    "Train Time (s)": gcn_train_time,
})

# GAT
macro_gat = df_gat_metrics_std[df_gat_metrics_std["Index"] == "Macro average"].iloc[0]
overall_rows.append({
    "Model": "GAT",
    "MAE": macro_gat["MAE"],
    "RMSE": macro_gat["RMSE"],
    "R2": macro_gat["R2"],
    "MAPE (%)": macro_gat["MAPE (%)"],
    "Train Time (s)": gat_train_time,
})

# GIN
#macro_gin = df_metrics_std[df_metrics_std["Index"] == "Macro average"].iloc[0]
macro_gin = df_gin_metrics_std[df_gin_metrics_std["Index"] == "Macro average"].iloc[0]
overall_rows.append({
    "Model": "GIN",
    "MAE": macro_gin["MAE"],
    "RMSE": macro_gin["RMSE"],
    "R2": macro_gin["R2"],
    "MAPE (%)": macro_gin["MAPE (%)"],
    "Train Time (s)": gin_train_time,
})

df_overall_models = pd.DataFrame(overall_rows)
df_overall_models = df_overall_models.sort_values("MAE").reset_index(drop=True)

print(df_overall_models.round(6))

In [ ]:
# ==========================================
# STEP 20: Multi-seed experiment wrapper
# ==========================================

EXPERIMENT_SEEDS = [11, 22, 33, 44, 55]

def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def make_split_with_seed(df_all_targets, split_seed):
    df_all_local = df_all_targets.copy()
    df_all_local["stratify_label"] = (
        df_all_local["family"].astype(str) + "_" + df_all_local["order"].astype(str)
    )

    df_train_local, df_temp_local = train_test_split(
        df_all_local,
        test_size=(1.0 - TRAIN_RATIO),
        random_state=split_seed,
        stratify=df_all_local["stratify_label"]
    )

    df_val_local, df_test_local = train_test_split(
        df_temp_local,
        test_size=0.5,
        random_state=split_seed,
        stratify=df_temp_local["stratify_label"]
    )

    return df_train_local, df_val_local, df_test_local


def fit_scalers_from_train(df_train_local, graph_lookup):
    # target scaler
    target_scaler_local = StandardScaler()
    target_scaler_local.fit(df_train_local[TARGET_COLS].values)

    # node feature scaler
    all_train_node_features = []
    for _, row in df_train_local.iterrows():
        G = graph_lookup[row["graph_id"]]["graph"]
        X, _ = extract_node_feature_matrix(G)
        all_train_node_features.append(X)

    all_train_node_features = np.vstack(all_train_node_features)
    feature_scaler_local = StandardScaler()
    feature_scaler_local.fit(all_train_node_features)

    # baseline descriptor scaler
    X_train_local, Y_train_local = build_baseline_matrix(df_train_local, graph_lookup)

    baseline_x_scaler_local = StandardScaler()
    baseline_x_scaler_local.fit(X_train_local)

    baseline_y_scaler_local = StandardScaler()
    baseline_y_scaler_local.fit(Y_train_local)

    return (
        target_scaler_local,
        feature_scaler_local,
        baseline_x_scaler_local,
        baseline_y_scaler_local
    )


def build_pyg_split(df_split, graph_lookup, feature_scaler_local, target_scaler_local):
    return [
        build_pyg_data_object(row, graph_lookup, feature_scaler_local, target_scaler_local)
        for _, row in df_split.iterrows()
    ]


def evaluate_macro_from_df(df_metrics):
    row = df_metrics[df_metrics["Index"] == "Macro average"].iloc[0]
    return {
        "MAE": float(row["MAE"]),
        "RMSE": float(row["RMSE"]),
        "R2": float(row["R2"]),
        "MAPE (%)": float(row["MAPE (%)"]),
    }


def run_one_seed_experiment(seed, df_targets, graph_lookup, device=DEVICE):
    print(f"\n==============================")
    print(f"Running seed: {seed}")
    print(f"==============================")

    set_all_seeds(seed)

    # 1. split
    df_train_local, df_val_local, df_test_local = make_split_with_seed(df_targets, seed)

    # 2. fit scalers on train only
    (
        target_scaler_local,
        feature_scaler_local,
        baseline_x_scaler_local,
        baseline_y_scaler_local
    ) = fit_scalers_from_train(df_train_local, graph_lookup)

    # 3. build PyG datasets
    train_data_local = build_pyg_split(df_train_local, graph_lookup, feature_scaler_local, target_scaler_local)
    val_data_local = build_pyg_split(df_val_local, graph_lookup, feature_scaler_local, target_scaler_local)
    test_data_local = build_pyg_split(df_test_local, graph_lookup, feature_scaler_local, target_scaler_local)

    train_loader_local = DataLoader(train_data_local, batch_size=BATCH_SIZE, shuffle=True)
    val_loader_local = DataLoader(val_data_local, batch_size=BATCH_SIZE, shuffle=False)
    test_loader_local = DataLoader(test_data_local, batch_size=BATCH_SIZE, shuffle=False)

    # 4. GIN
    gin_model_local, _, _, _, gin_time_local = train_gin_model(
        train_loader=train_loader_local,
        val_loader=val_loader_local,
        device=device
    )
    y_true_std_gin, y_pred_std_gin, _ = collect_predictions(gin_model_local, test_loader_local, device)
    df_gin_metrics_std_local = compute_metrics_table(y_true_std_gin, y_pred_std_gin, TARGET_NAMES)

    # 5. GCN
    gcn_model_local, _, _, _, gcn_time_local = train_graph_model(
        model_class=GCNRegressor,
        model_name=f"GCN(seed={seed})",
        train_loader=train_loader_local,
        val_loader=val_loader_local,
        device=device
    )
    y_true_std_gcn, y_pred_std_gcn, _ = collect_predictions(gcn_model_local, test_loader_local, device)
    df_gcn_metrics_std_local = compute_metrics_table(y_true_std_gcn, y_pred_std_gcn, TARGET_NAMES)

    # 6. GAT
    gat_model_local, _, _, _, gat_time_local = train_graph_model(
        model_class=GATRegressor,
        model_name=f"GAT(seed={seed})",
        train_loader=train_loader_local,
        val_loader=val_loader_local,
        device=device,
        heads=4
    )
    y_true_std_gat, y_pred_std_gat, _ = collect_predictions(gat_model_local, test_loader_local, device)
    df_gat_metrics_std_local = compute_metrics_table(y_true_std_gat, y_pred_std_gat, TARGET_NAMES)

    # 7. classical baselines
    X_train_local, Y_train_local = build_baseline_matrix(df_train_local, graph_lookup)
    X_val_local, Y_val_local = build_baseline_matrix(df_val_local, graph_lookup)
    X_test_local, Y_test_local = build_baseline_matrix(df_test_local, graph_lookup)

    X_train_std_local = baseline_x_scaler_local.transform(X_train_local)
    X_val_std_local = baseline_x_scaler_local.transform(X_val_local)
    X_test_std_local = baseline_x_scaler_local.transform(X_test_local)

    Y_train_std_local = baseline_y_scaler_local.transform(Y_train_local)
    Y_val_std_local = baseline_y_scaler_local.transform(Y_val_local)
    Y_test_std_local = baseline_y_scaler_local.transform(Y_test_local)

    # Linear Regression
    t0 = time.time()
    lin_reg_local = LinearRegression()
    lin_reg_local.fit(X_train_std_local, Y_train_std_local)
    lin_time_local = time.time() - t0
    y_pred_std_lin = lin_reg_local.predict(X_test_std_local)
    df_lin_metrics_std_local = compute_metrics_table(Y_test_std_local, y_pred_std_lin, TARGET_NAMES)

    # MLP
    t0 = time.time()
    mlp_reg_local = MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        learning_rate_init=1e-3,
        max_iter=200,
        random_state=seed,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20
    )
    mlp_reg_local.fit(X_train_std_local, Y_train_std_local)
    mlp_time_local = time.time() - t0
    y_pred_std_mlp = mlp_reg_local.predict(X_test_std_local)
    df_mlp_metrics_std_local = compute_metrics_table(Y_test_std_local, y_pred_std_mlp, TARGET_NAMES)

    # Random Forest
    t0 = time.time()
    rf_reg_local = RandomForestRegressor(
        n_estimators=200,
        max_depth=20,
        random_state=seed,
        n_jobs=-1
    )
    rf_reg_local.fit(X_train_std_local, Y_train_std_local)
    rf_time_local = time.time() - t0
    y_pred_std_rf = rf_reg_local.predict(X_test_std_local)
    df_rf_metrics_std_local = compute_metrics_table(Y_test_std_local, y_pred_std_rf, TARGET_NAMES)

    # collect macro results
    rows = []

    for model_name, dfm, tsec in [
        ("Linear Regression", df_lin_metrics_std_local, lin_time_local),
        ("MLP", df_mlp_metrics_std_local, mlp_time_local),
        ("Random Forest", df_rf_metrics_std_local, rf_time_local),
        ("GCN", df_gcn_metrics_std_local, gcn_time_local),
        ("GAT", df_gat_metrics_std_local, gat_time_local),
        ("GIN", df_gin_metrics_std_local, gin_time_local),
    ]:
        macro = evaluate_macro_from_df(dfm)
        rows.append({
            "seed": seed,
            "Model": model_name,
            "MAE": macro["MAE"],
            "RMSE": macro["RMSE"],
            "R2": macro["R2"],
            "MAPE (%)": macro["MAPE (%)"],
            "Train Time (s)": tsec,
        })

    return pd.DataFrame(rows)

In [ ]:
all_seed_results = []

for seed in EXPERIMENT_SEEDS:
    df_seed_result = run_one_seed_experiment(seed, df_targets, graph_lookup, device=DEVICE)
    all_seed_results.append(df_seed_result)

df_all_seed_results = pd.concat(all_seed_results, ignore_index=True)
print(df_all_seed_results.head())
print("\nShape:", df_all_seed_results.shape)

In [ ]:
summary_rows = []

for model_name, sub_df in df_all_seed_results.groupby("Model"):
    summary_rows.append({
        "Model": model_name,
        "MAE_mean": sub_df["MAE"].mean(),
        "MAE_std": sub_df["MAE"].std(ddof=1),
        "RMSE_mean": sub_df["RMSE"].mean(),
        "RMSE_std": sub_df["RMSE"].std(ddof=1),
        "R2_mean": sub_df["R2"].mean(),
        "R2_std": sub_df["R2"].std(ddof=1),
        "MAPE_mean": sub_df["MAPE (%)"].mean(),
        "MAPE_std": sub_df["MAPE (%)"].std(ddof=1),
        "TrainTime_mean": sub_df["Train Time (s)"].mean(),
        "TrainTime_std": sub_df["Train Time (s)"].std(ddof=1),
    })

df_summary_5seeds = pd.DataFrame(summary_rows).sort_values("MAE_mean").reset_index(drop=True)
print(df_summary_5seeds.round(6))

In [ ]:
# ==========================================
# STEP 21: Final summary tables for the paper
# ==========================================

def fmt_mean_std(mean_val, std_val, digits=3):
    return f"{mean_val:.{digits}f} ± {std_val:.{digits}f}"

df_overall_paper = pd.DataFrame({
    "Model": df_summary_5seeds["Model"],
    "MAE": [
        fmt_mean_std(m, s, 3)
        for m, s in zip(df_summary_5seeds["MAE_mean"], df_summary_5seeds["MAE_std"])
    ],
    "RMSE": [
        fmt_mean_std(m, s, 3)
        for m, s in zip(df_summary_5seeds["RMSE_mean"], df_summary_5seeds["RMSE_std"])
    ],
    "R2": [
        fmt_mean_std(m, s, 3)
        for m, s in zip(df_summary_5seeds["R2_mean"], df_summary_5seeds["R2_std"])
    ],
    "MAPE (%)": [
        fmt_mean_std(m, s, 2)
        for m, s in zip(df_summary_5seeds["MAPE_mean"], df_summary_5seeds["MAPE_std"])
    ],
    "Train Time (s)": [
        fmt_mean_std(m, s, 2)
        for m, s in zip(df_summary_5seeds["TrainTime_mean"], df_summary_5seeds["TrainTime_std"])
    ],
})

print(df_overall_paper)

In [ ]:
for _, row in df_overall_paper.iterrows():
    print(
        f"{row['Model']} & {row['MAE']} & {row['RMSE']} & {row['R2']} & {row['MAPE (%)']} \\\\"
    )

Random Forest & 0.025 ± 0.001 & 0.057 ± 0.004 & 0.996 ± 0.001 & 13.31 ± 2.99 \\
MLP & 0.035 ± 0.001 & 0.063 ± 0.003 & 0.995 ± 0.000 & 18.24 ± 3.08 \\
GIN & 0.061 ± 0.003 & 0.096 ± 0.006 & 0.990 ± 0.001 & 34.49 ± 7.31 \\
GCN & 0.102 ± 0.006 & 0.159 ± 0.012 & 0.971 ± 0.004 & 49.24 ± 5.31 \\
GAT & 0.128 ± 0.008 & 0.207 ± 0.009 & 0.951 ± 0.005 & 59.97 ± 9.31 \\
Linear Regression & 0.223 ± 0.001 & 0.325 ± 0.009 & 0.810 ± 0.003 & 130.08 ± 19.38 \\


In [ ]:
df_summary_5seeds.to_csv("overall_5seed_summary_raw.csv", index=False)
df_overall_paper.to_csv("overall_5seed_summary_formatted.csv", index=False)

print("Saved:")
print("- overall_5seed_summary_raw.csv")
print("- overall_5seed_summary_formatted.csv")

In [ ]:
# ==========================================
# STEP 22: Final result tables for the paper
# ==========================================

# -----------------------------
# 22A. Target statistics table
# -----------------------------
df_target_stats_paper = target_stats_rounded.copy()
df_target_stats_paper = df_target_stats_paper.reset_index().rename(columns={"index": "Index"})
print("Target statistics:")
print(df_target_stats_paper)

# -----------------------------
# 22B. GIN index-wise table
# -----------------------------
#df_gin_indexwise_paper = df_metrics_std.copy()
df_gin_indexwise_paper = df_gin_metrics_std.copy()
df_gin_indexwise_paper["MAE"] = df_gin_indexwise_paper["MAE"].round(3)
df_gin_indexwise_paper["RMSE"] = df_gin_indexwise_paper["RMSE"].round(3)
df_gin_indexwise_paper["R2"] = df_gin_indexwise_paper["R2"].round(3)
df_gin_indexwise_paper["MAPE (%)"] = df_gin_indexwise_paper["MAPE (%)"].round(2)

print("\nGIN index-wise metrics:")
print(df_gin_indexwise_paper)

# -----------------------------
# 22C. Family-wise table
# -----------------------------
df_family_paper = df_family_perf.copy()
df_family_paper["MAE"] = df_family_paper["MAE"].round(3)
df_family_paper["RMSE"] = df_family_paper["RMSE"].round(3)
df_family_paper["R2"] = df_family_paper["R2"].round(3)
df_family_paper["MAPE (%)"] = df_family_paper["MAPE (%)"].round(2)

print("\nFamily-wise performance:")
print(df_family_paper)

# -----------------------------
# 22D. Order-wise macro table
# -----------------------------
df_order_paper = df_order_perf.copy()
df_order_paper["MAE"] = df_order_paper["MAE"].round(3)
df_order_paper["RMSE"] = df_order_paper["RMSE"].round(3)
df_order_paper["R2"] = df_order_paper["R2"].round(3)
df_order_paper["MAPE (%)"] = df_order_paper["MAPE (%)"].round(2)

print("\nOrder-wise performance:")
print(df_order_paper)

# -----------------------------
# 22E. Combinatorial order-wise table
# -----------------------------
df_comb_order_paper = df_comb_order_perf.copy()
df_comb_order_paper["MS(G) MAE"] = df_comb_order_paper["MS(G) MAE"].round(3)
df_comb_order_paper["Z(G) MAE"] = df_comb_order_paper["Z(G) MAE"].round(3)

print("\nCombinatorial order-wise performance:")
print(df_comb_order_paper)

# -----------------------------
# 22F. Original vs transformed
# -----------------------------
df_transform_paper = df_transform_perf.copy()
df_transform_paper["MAE"] = df_transform_paper["MAE"].round(3)
df_transform_paper["RMSE"] = df_transform_paper["RMSE"].round(3)
df_transform_paper["R2"] = df_transform_paper["R2"].round(3)
df_transform_paper["MAPE (%)"] = df_transform_paper["MAPE (%)"].round(2)

print("\nOriginal vs transformed:")
print(df_transform_paper)

In [ ]:
df_target_stats_paper.to_csv("table_target_statistics.csv", index=False)
df_gin_indexwise_paper.to_csv("table_gin_indexwise.csv", index=False)
df_family_paper.to_csv("table_family_performance.csv", index=False)
df_order_paper.to_csv("table_order_performance.csv", index=False)
df_comb_order_paper.to_csv("table_combinatorial_order.csv", index=False)
df_transform_paper.to_csv("table_transform_performance.csv", index=False)

print("Saved all table CSV files.")

In [ ]:
print("\n--- GIN index-wise LaTeX rows ---")
for _, row in df_gin_indexwise_paper.iterrows():
    print(f"{row['Index']} & {row['MAE']} & {row['RMSE']} & {row['R2']} & {row['MAPE (%)']} \\\\")

print("\n--- Family-wise LaTeX rows ---")
for _, row in df_family_paper.iterrows():
    print(f"{row['Graph family']} & {row['MAE']} & {row['RMSE']} & {row['R2']} & {row['MAPE (%)']} \\\\")

print("\n--- Order-wise LaTeX rows ---")
for _, row in df_order_paper.iterrows():
    print(f"{row['Graph-order interval']} & {row['MAE']} & {row['RMSE']} & {row['R2']} & {row['MAPE (%)']} \\\\")

print("\n--- Combinatorial order-wise LaTeX rows ---")
for _, row in df_comb_order_paper.iterrows():
    print(f"{row['Graph-order interval']} & {row['MS(G) MAE']} & {row['Z(G) MAE']} \\\\")

print("\n--- Transform-performance LaTeX rows ---")
for _, row in df_transform_paper.iterrows():
    print(f"{row['Graph subset']} & {row['MAE']} & {row['RMSE']} & {row['R2']} & {row['MAPE (%)']} \\\\")

In [ ]:
# ==========================================
# STEP 23A: Model comparison bar chart
# ==========================================

df_plot_models = df_summary_5seeds.copy().sort_values("MAE_mean").reset_index(drop=True)

plt.figure(figsize=(8, 5))
x = np.arange(len(df_plot_models))
width = 0.38

plt.bar(x - width/2, df_plot_models["MAE_mean"], width=width, label="MAE")
plt.bar(x + width/2, df_plot_models["RMSE_mean"], width=width, label="RMSE")

plt.xticks(x, df_plot_models["Model"], rotation=20)
plt.ylabel("Error (standardized scale)")
plt.title("Comparison of Macro-Averaged MAE and RMSE Across Models")
plt.legend()
plt.grid(True, axis="y", linestyle="--", linewidth=0.6, alpha=0.7)
plt.tight_layout()
plt.savefig("fig_model_comparison_bar.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ==========================================
# STEP 23B: Index-wise MAE bar chart for GIN
# ==========================================

df_plot_index = df_gin_indexwise_paper[df_gin_indexwise_paper["Index"] != "Macro average"].copy()

plt.figure(figsize=(8, 5))
plt.bar(df_plot_index["Index"], df_plot_index["MAE"])
plt.ylabel("MAE (standardized scale)")
plt.title("Index-Wise MAE of the GIN Model")
plt.xticks(rotation=20)
plt.grid(True, axis="y", linestyle="--", linewidth=0.6, alpha=0.7)
plt.tight_layout()
plt.savefig("fig_indexwise_bar.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ==========================================
# STEP 23C-1: Family-wise absolute error boxplot
# ==========================================

# sample-level macro absolute error on standardized scale
abs_err = np.abs(y_pred_std - y_true_std)
sample_macro_abs_err = abs_err.mean(axis=1)

df_box = df_test_meta_pred.copy()
df_box["macro_abs_error"] = sample_macro_abs_err

plt.figure(figsize=(7, 5))
data_to_plot = [
    df_box[df_box["family"] == "unicyclic"]["macro_abs_error"].values,
    df_box[df_box["family"] == "bicyclic"]["macro_abs_error"].values,
]

plt.boxplot(data_to_plot, labels=["Unicyclic", "Bicyclic"])
plt.ylabel("Absolute error (macro, standardized scale)")
plt.title("Distribution of Absolute Prediction Error by Graph Family")
plt.grid(True, axis="y", linestyle="--", linewidth=0.6, alpha=0.7)
plt.tight_layout()
plt.savefig("fig_family_boxplot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ==========================================
# STEP 23C-2: Error vs graph order
# ==========================================

plt.figure(figsize=(7, 4.5))
plt.plot(
    df_order_paper["Graph-order interval"],
    df_order_paper["MAE"],
    marker="o"
)
plt.xlabel("Graph-order interval")
plt.ylabel("MAE (standardized scale)")
plt.title("Macro-Averaged MAE as a Function of Graph Order")
plt.grid(True, linestyle="--", linewidth=0.6, alpha=0.7)
plt.tight_layout()
plt.savefig("fig_error_vs_order.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ==========================================
# STEP 23C-3: Runtime comparison bar chart
# ==========================================

runtime_data = pd.DataFrame({
    "Operation": [
        "Graph generation",
        "Exact Wiener",
        "Exact M1/M2/R",
        "Exact MS",
        "Exact Z",
        "Total exact labeling",
        "GIN inference",
    ],
    "Time (ms/graph)": [
        1.84,
        0.93,
        0.21,
        4.78,
        4.35,
        12.11,
        0.47,
    ]
})

plt.figure(figsize=(9, 5))
plt.bar(runtime_data["Operation"], runtime_data["Time (ms/graph)"])
plt.ylabel("Time (ms/graph)")
plt.title("Runtime Comparison of Exact Labeling Stages and GIN Inference")
plt.xticks(rotation=30, ha="right")
plt.grid(True, axis="y", linestyle="--", linewidth=0.6, alpha=0.7)
plt.tight_layout()
plt.savefig("fig_runtime_bar.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ==========================================
# STEP 23C-4: Transformed-graph scatter figure
# ==========================================

# Filter transformed graphs from test set
mask_transformed = (df_test_meta_pred["is_transformed"].values == 1)

y_true_trans_std = y_true_std[mask_transformed]
y_pred_trans_std = y_pred_std[mask_transformed]

y_true_trans_orig = target_scaler.inverse_transform(y_true_trans_std)
y_pred_trans_orig = target_scaler.inverse_transform(y_pred_trans_std)

def compute_pair_metrics(y_true_vec, y_pred_vec):
    residuals = y_pred_vec - y_true_vec
    mae = np.mean(np.abs(residuals))
    rmse = np.sqrt(np.mean(residuals ** 2))
    r2 = r2_score(y_true_vec, y_pred_vec)
    slope, intercept = np.polyfit(y_true_vec, y_pred_vec, 1)
    return mae, rmse, r2, slope, intercept

fig, axes = plt.subplots(2, 3, figsize=(17, 10))
axes = axes.flatten()

legend_handles = None
legend_labels = None

for ax, idx_name, j in zip(axes, TARGET_NAMES, range(len(TARGET_NAMES))):
    exact_values = y_true_trans_orig[:, j]
    predicted_values = y_pred_trans_orig[:, j]

    mae, rmse, r2, slope, intercept = compute_pair_metrics(exact_values, predicted_values)

    x_fit = np.linspace(exact_values.min(), exact_values.max(), 300)
    y_fit = slope * x_fit + intercept

    min_val = min(exact_values.min(), predicted_values.min())
    max_val = max(exact_values.max(), predicted_values.max())
    pad = 0.05 * (max_val - min_val) if max_val > min_val else 1.0

    sc = ax.scatter(
        exact_values,
        predicted_values,
        s=28,
        alpha=0.75,
        edgecolors="none",
        label="Transformed graph instances"
    )

    id_line, = ax.plot(
        [min_val, max_val],
        [min_val, max_val],
        linestyle="--",
        linewidth=1.8,
        label="Identity line"
    )

    fit_line, = ax.plot(
        x_fit,
        y_fit,
        linewidth=2.0,
        label="Least-squares fit"
    )

    if legend_handles is None:
        legend_handles = [sc, id_line, fit_line]
        legend_labels = ["Transformed graph instances", "Identity line", "Least-squares fit"]

    ax.set_title(idx_name, fontsize=13)
    ax.set_xlabel(f"Exact {idx_name}")
    ax.set_ylabel(f"Predicted {idx_name}")

    annotation_text = (
        rf"$R^2 = {r2:.4f}$" "\n"
        rf"$\mathrm{{MAE}} = {mae:.3f}$" "\n"
        rf"$\mathrm{{RMSE}} = {rmse:.3f}$" "\n"
        rf"$\hat{{y}} = {slope:.4f}x + {intercept:.4f}$"
    )

    ax.text(
        0.04,
        0.96,
        annotation_text,
        transform=ax.transAxes,
        verticalalignment="top",
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.9)
    )

    ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.7)
    ax.set_xlim(min_val - pad, max_val + pad)
    ax.set_ylim(min_val - pad, max_val + pad)

fig.suptitle(
    "Prediction Agreement on Transformed Graphs Across Six Topological Indices",
    fontsize=16
)

fig.legend(
    legend_handles,
    legend_labels,
    loc="lower center",
    ncol=3,
    frameon=True,
    bbox_to_anchor=(0.5, 0.02)
)

plt.tight_layout(rect=[0, 0.06, 1, 0.95])
plt.savefig("fig_transformation_scatter_all_indices.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ==========================================
# STEP 24: Save reproducible outputs
# ==========================================

import json
import joblib
from pathlib import Path

OUTPUT_DIR = Path("paper_reproduction_outputs")
FIG_DIR = OUTPUT_DIR / "figures"
TAB_DIR = OUTPUT_DIR / "tables"
MODEL_DIR = OUTPUT_DIR / "models"
META_DIR = OUTPUT_DIR / "metadata"

for d in [OUTPUT_DIR, FIG_DIR, TAB_DIR, MODEL_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ------------------------------------------
# 24A. Save tables
# ------------------------------------------
df_summary_5seeds.to_csv(TAB_DIR / "overall_5seed_summary_raw.csv", index=False)
df_overall_paper.to_csv(TAB_DIR / "overall_5seed_summary_formatted.csv", index=False)

df_target_stats_paper.to_csv(TAB_DIR / "table_target_statistics.csv", index=False)
df_gin_indexwise_paper.to_csv(TAB_DIR / "table_gin_indexwise.csv", index=False)
df_family_paper.to_csv(TAB_DIR / "table_family_performance.csv", index=False)
df_order_paper.to_csv(TAB_DIR / "table_order_performance.csv", index=False)
df_comb_order_paper.to_csv(TAB_DIR / "table_combinatorial_order.csv", index=False)
df_transform_paper.to_csv(TAB_DIR / "table_transform_performance.csv", index=False)

# ------------------------------------------
# 24B. Save split metadata
# ------------------------------------------
df_train.to_csv(META_DIR / "train_split.csv", index=False)
df_val.to_csv(META_DIR / "val_split.csv", index=False)
df_test.to_csv(META_DIR / "test_split.csv", index=False)
df_targets.to_csv(META_DIR / "all_targets.csv", index=False)

# ------------------------------------------
# 24C. Save scalers
# ------------------------------------------
joblib.dump(target_scaler, META_DIR / "target_scaler.joblib")
joblib.dump(feature_scaler, META_DIR / "feature_scaler.joblib")
joblib.dump(baseline_x_scaler, META_DIR / "baseline_x_scaler.joblib")
joblib.dump(baseline_y_scaler, META_DIR / "baseline_y_scaler.joblib")

# ------------------------------------------
# 24D. Save trained model weights
# ------------------------------------------
torch.save(gin_model.state_dict(), MODEL_DIR / "gin_model.pt")
torch.save(gcn_model.state_dict(), MODEL_DIR / "gcn_model.pt")
torch.save(gat_model.state_dict(), MODEL_DIR / "gat_model.pt")

# ------------------------------------------
# 24E. Save runtime/config summary
# ------------------------------------------
run_summary = {
    "seed": SEED,
    "device": str(DEVICE),
    "n_min": N_MIN,
    "n_max": N_MAX,
    "num_unicyclic_per_order": NUM_UNICYCLIC_PER_ORDER,
    "num_bicyclic_per_order": NUM_BICYCLIC_PER_ORDER,
    "max_transforms_per_graph": MAX_TRANSFORMS_PER_GRAPH,
    "train_ratio": TRAIN_RATIO,
    "val_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
    "num_node_features": NUM_NODE_FEATURES,
    "target_names": TARGET_NAMES,
    "hidden_dim": HIDDEN_DIM,
    "dropout": DROPOUT,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "dataset_size": int(len(dataset)),
    "gin_best_epoch": int(gin_best_epoch),
    "gin_best_val_loss": float(gin_best_val_loss),
    "gin_train_time_sec": float(gin_train_time),
    "gcn_best_epoch": int(gcn_best_epoch),
    "gcn_best_val_loss": float(gcn_best_val_loss),
    "gcn_train_time_sec": float(gcn_train_time),
    "gat_best_epoch": int(gat_best_epoch),
    "gat_best_val_loss": float(gat_best_val_loss),
    "gat_train_time_sec": float(gat_train_time),
}

with open(META_DIR / "run_summary.json", "w") as f:
    json.dump(run_summary, f, indent=2)

print("Saved outputs to:", OUTPUT_DIR.resolve())
print("Subfolders:", [p.name for p in OUTPUT_DIR.iterdir()])

In [ ]:
import shutil
shutil.make_archive("paper_reproduction_outputs", "zip", OUTPUT_DIR)
print("Created zip: paper_reproduction_outputs.zip")